In [4]:
# esta rotina atualiza o campo resumo_recurso da tabela anterioridades_desc
# portanto primeiro deve ser rodado insert_anterioridades_desc.ipynb para ter esta tabela atualizada
# esta rotina consulta a tabela de carga do sinergias e depois consulta a despachos_pag para encontrar a petição 214
# esta petição é localizada no disco local da máquina feito o OCR, anonimizada e depois enviada por prompt a llM
# query = f"Resuma os principais argumentos apresentados pela requerente em sua petição recursal {texto_anon}"
# o resultado é salvo no campo resumo_recurso da tabela anterioridades_desc

# pip install mysql-connector-python
import mysql.connector

conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

In [5]:
import pandas as pd 

numero = "PI0808715"
# atualiza no localhost as tres tabelas: carga, anterioridades e anterioridades_desc
comando = f"SELECT * FROM arquivados WHERE numero='{numero}'"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
print(resultado)

[(1011532, '1.1', 'PI0808715', datetime.date(2011, 8, 9), 'dialp', 0, 0), (1475660, '1.3', 'PI0808715', datetime.date(2014, 8, 12), 'dialp', 0, 0), (1501853, '6.6', 'PI0808715', datetime.date(2014, 9, 16), 'dialp', 0, 0), (2788968, '7.1', 'PI0808715', datetime.date(2017, 4, 18), 'dialp', 0, 1), (2790019, '15.11', 'PI0808715', datetime.date(2017, 4, 18), 'dialp', 0, 0), (2956330, '9.2', 'PI0808715', datetime.date(2017, 9, 19), 'dialp', 0, 2), (3015281, '12.2', 'PI0808715', datetime.date(2017, 12, 19), 'dialp', 0, 0)]


In [6]:
import json
import requests

def conectar_siscap(url,return_json=False):
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url,headers=headers,verify=False)
    if response.status_code == 200:
        if return_json:
            data = response.json()
            json_data = json.dumps(data, indent=4)
            return(json_data)
        else:
            return response.text
    else:
        return(f"Erro: {response.status_code}")

In [8]:
# conecte na VPN
url = 'https://siscap.inpi.gov.br/adm/pareceres/dicel/1120120181571338559.txt' # 1 doc
numero='112012018157'
codigo = '1338559'
divisao = 'dicel'
url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
print(url)
texto_relatorio = conectar_siscap(url,return_json=False)
print(texto_relatorio)
caminho_do_arquivo=f"pareceres/{divisao}/{numero}{codigo}.txt"
with open(caminho_do_arquivo, 'w', encoding='utf-8') as arquivo:
    arquivo.write(texto_relatorio)

https://siscap.inpi.gov.br/adm/pareceres/dicel/1120120181571338559.txt
                                      SERVIÇO PÚBLICO FEDERAL
                                       MINISTÉRIO DA ECONOMIA
                           INSTITUTO NACIONAL DA PROPRIEDADE INDUSTRIAL

                                             RELATÓRIO DE EXAME TÉCNICO

 N.° do Pedido:                      BR112012018157-2         N.° de Depósito PCT:US11/021793
 Data de Depósito:                   20/01/2011
 Prioridade Unionista:               US 12/692080 (22/01/2010)
 Depositante:                        KONINKLIJKE PHILIPS N.V (NL)
 Inventor:                           Roger J. Quy
 Título:                             Sistema para monitorar exercício, método para monitorar um
                                     parâmetro de exercício usando um dispositivo de internet sem fio e
                                     meio de armazenamento em memória 

                                                                

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'siscap.inpi.gov.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [9]:
import re
import hashlib
from collections import defaultdict


class DataAnonymizer:
    def __init__(self):
        # token -> valor original
        self.token_map = {}

        # valor original -> token (garante determinismo)
        self.reverse_map = {}

        # contadores por tipo
        self.token_counter = defaultdict(int)

    # ==============================
    # GERAÇÃO DE TOKEN
    # ==============================
    def _generate_token(self, tipo, valor):
        if not valor:
            return valor

        valor = valor.strip()

        # Determinístico: mesma string → mesmo token
        if valor in self.reverse_map:
            return self.reverse_map[valor]

        self.token_counter[tipo] += 1
        token = f"[{tipo}_{self.token_counter[tipo]}]"

        self.token_map[token] = valor
        self.reverse_map[valor] = token

        return token

    # ==============================
    # REMOÇÃO DE CABEÇALHOS
    # ==============================
    def remover_cabecalhos(self, texto):
        padroes_remover = [
            r"Assinado digitalmente por.*",
            r"Documento assinado eletronicamente.*",
            r"Protocolo:\s*\d+",
            r"URL para download:.*",
            r"Hash de autenticação:.*",
        ]

        for padrao in padroes_remover:
            texto = re.sub(padrao, "", texto, flags=re.IGNORECASE)

        return texto

    # ==============================
    # ANONIMIZAÇÃO DE CPFs
    # ==============================
    def anonymize_cpfs(self, texto):
        regex_cpf = r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b"

        def substituir(match):
            cpf = match.group()
            return self._generate_token("CPF", cpf)

        return re.sub(regex_cpf, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE CNPJ
    # ==============================
    def anonymize_cnpj(self, texto):
        regex_cnpj = r"\b\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}\b"

        def substituir(match):
            cnpj = match.group()
            return self._generate_token("CNPJ", cnpj)

        return re.sub(regex_cnpj, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE PROCESSOS (9 dígitos)
    # ==============================
    def anonymize_processos(self, texto):
        regex_processo = r"\b\d{9}\b"

        def substituir(match):
            processo = match.group()
            return self._generate_token("PROCESSO", processo)

        return re.sub(regex_processo, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE EMAIL
    # ==============================
    def anonymize_emails(self, texto):
        regex_email = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

        def substituir(match):
            email = match.group()
            return self._generate_token("EMAIL", email)

        return re.sub(regex_email, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE NOMES SIMPLES (heurística)
    # ==============================
    def anonymize_nomes_maiusculos(self, texto):
        # Heurística: nomes em caixa alta com pelo menos 2 palavras
        regex_nome = r"\b([A-ZÁÉÍÓÚÂÊÔÃÕÇ]{2,}(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ]{2,})+)\b"

        def substituir(match):
            nome = match.group(1)
            return self._generate_token("PESSOA_NATURAL", nome)

        return re.sub(regex_nome, substituir, texto)

    # ==============================
    # DOCUMENTOS GENÉRICOS DE IDENTIFICAÇÃO
    # ==============================
    def anonymize_documentos_identificacao(self, texto):
        regex = r"(CPF\/CNPJ|CPF|CNPJ)\s*:\s*([A-Z0-9\-\.\/]+)"
    
        def substituir(match):
            rotulo = match.group(1)
            valor = match.group(2)
            token = self._generate_token("DOC_ID", valor)
            return f"{rotulo}: {token}"
    
        return re.sub(regex, substituir, texto, flags=re.IGNORECASE)
        
    # ==============================
    # IDENTIFICADORES INTERNACIONAIS
    # ==============================
    def anonymize_identificadores_internacionais(self, texto):
        regex = r"\b[A-Z]{2}\d{6,}\b"
    
        def substituir(match):
            valor = match.group()
            return self._generate_token("REGISTRO_INT", valor)
    
        return re.sub(regex, substituir, texto)

    # ==============================
    # ENDEREÇO
    # ==============================
    def anonymize_endereco(self, texto):
        regex = r"(Endere[cç]o\s*:\s*)(.+)"
    
        def substituir(match):
            prefixo = match.group(1)   # "Endereço: "
            valor = match.group(2).strip()
    
            token = self._generate_token("ENDERECO", valor)
    
            return f"{prefixo}{token}"
    
        return re.sub(regex, substituir, texto, flags=re.IGNORECASE)

    # ==============================
    # REMOVER A LINHA INTEIRA QUE TENHA CEP
    # ==============================
    def anonymize_remover_linhas_com_cep(self, texto):
        linhas = texto.splitlines()
    
        linhas_filtradas = [
            linha for linha in linhas
            if not re.search(r'\bCEP\b', linha, flags=re.IGNORECASE)
        ]
    
        return "\n".join(linhas_filtradas)
    
    # ==============================
    # REMOVER CABEÇALHO DAS PÁGINAS DA PETIÇÃO
    # ==============================
    def anonymize_remover_cabecalhos_pagina(self, texto):
        linhas = texto.splitlines()
    
        linhas_filtradas = [
            linha for linha in linhas
            if not re.search(r'^\s*Peticao\b', linha, flags=re.IGNORECASE)
        ]
    
        return "\n".join(linhas_filtradas)

    # ==============================
    # TELEFONE
    # ==============================
    def anonymize_telefone(self, texto):
        regex = r"(Telefone|Fax\s*:\s*)(.+)"
        regex = r"\b(Fone\/Fax|Fone|Telefone|Tel\.?|Fax)\b\s*:?\s*([\(\d][\d\.\-\)\s]+)"
    
        def substituir(match):
            prefixo = match.group(1)   # "Endereço: "
            valor = match.group(2).strip()
    
            token = self._generate_token("TELEFONE", valor)
    
            return f"{prefixo}{token}"
    
        return re.sub(regex, substituir, texto, flags=re.IGNORECASE)

    # ==============================
    # NOMES ROTULADOS
    # ==============================
    def anonymize_nomes_rotulados(self, texto):
        regex = r"(Requerente|Técnico|Inventor|Procurador)\s*:\s*([A-ZÁÉÍÓÚÂÊÔÃÕÇ\s]+)"
    
        def substituir(match):
            rotulo = match.group(1)
            nome = match.group(2).strip()
            token = self._generate_token("PESSOA_NATURAL", nome)
            return f"{rotulo}: {token}"
    
        return re.sub(regex, substituir, texto)
        
    # ==============================
    # PIPELINE COMPLETO
    # ==============================
    def anonymize_texto(self, texto):
        texto = self.remover_cabecalhos(texto)
        texto = self.anonymize_cpfs(texto)
        texto = self.anonymize_cnpj(texto)
        texto = self.anonymize_emails(texto)
        texto = self.anonymize_processos(texto)
        #texto = self.anonymize_nomes_maiusculos(texto) # este critério estava retirando qualquer sequencia de maiusculos o que elimina título do pedido
        texto = self.anonymize_documentos_identificacao(texto)
        #texto = self.anonymize_identificadores_internacionais(texto)
        texto = self.anonymize_endereco(texto)
        texto = self.anonymize_telefone(texto)
        texto = self.anonymize_nomes_rotulados(texto)
        texto = self.anonymize_remover_linhas_com_cep(texto)
        texto = self.anonymize_remover_cabecalhos_pagina(texto)

        return texto

    
    # ==============================
    # DESTOKENIZAÇÃO
    # ==============================
    def deanonymize(self, texto):
        # Substitui tokens pelos valores originais
        for token, valor in self.token_map.items():
            texto = texto.replace(token, valor)
        return texto

    def iniciar_apos_recurso(self, texto):
        padrao = r"(RECURSO\s+do\s+despacho\s+que\s+indeferiu\s+o\s+Pedido\s+de\s+Paten\s*te|Excelentissimo|Ilmo\s+Senhor\s+presidente|recurso\s+ao\s+presidente\s+|apresentar\s+recurso\s+desta\s+decis[aã]o|Em\s+resposta\s+a\s*(?:o|ao)\s+Indeferi\s*-?\s*mento)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            padrao = r"^\s*RECURSO\s+CONTRA\s+INDEFERIMENTO\s*$"
            match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
            if match:
                return texto[match.start():]
            else:
                padrao = r"^\s*RECURSO\s+AO\s+INDEFERIMENTO\s*$"
                match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
                if match:
                    return texto[match.start():]
                else:
                    padrao = r"^\s*INTERPOSICAO\s+DE\s+RECURSO\s+AO\s+INDEFERIMENTO\s*$"
                    match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
                    if match:
                        return texto[match.start():]
                    else:
                        return ""

    def iniciar_apos_recurso_alternativo(self, texto):
        padrao = r"(recurso\s+contra\s+o\s+indeferimento|recurso\s+ao\s+indeferimento|recurso\s+contra\s+decisao\s+de\s+indeferimento)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            padrao = r"(ilustrissimos\s+examinadores)"
            match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
            if match:
                return texto[match.start():]
            else:
                padrao = r"(recurso\s+que\s+bastante\s+faz)"
                match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
                if match:
                    return texto[match.start():]
                return ""
        
import re
import unicodedata

def normalizar(texto):
    texto = texto.replace('\xa0', ' ')  # remove non-breaking space
    texto = unicodedata.normalize('NFKD', texto)
    texto = texto.encode('ASCII', 'ignore').decode('ASCII')  # remove acentos
    #texto = re.sub(r'\s+', ' ', texto)  # colapsa múltiplos espaços/quebras
    return texto
        
documento = """
RECURSO do despacho que indeferiu o Pedido de Patente
Requerente: JOÃO SILVA OLIVEIRA CPF 123.456.789-00
Técnico: RICARDO FREDERICO NICOL
Processo anterior: 123456789
Email: joao@email.com
Assinado digitalmente por servidor INPI.
"""

anonymizer = DataAnonymizer()
documento = normalizar(documento)
documento = anonymizer.iniciar_apos_recurso(documento)
texto_anon = anonymizer.anonymize_texto(documento)
print("=== DOCUMENTO ANONIMIZADO ===")
if (texto_anon==''):
    texto_anom = anonymizer.iniciar_apos_recurso_alternativo(documento)
print(texto_anon)

=== DOCUMENTO ANONIMIZADO ===
RECURSO do despacho que indeferiu o Pedido de Patente
Requerente: [PESSOA_NATURAL_1][CPF_1]
Tecnico: RICARDO FREDERICO NICOL
Processo anterior: [PROCESSO_1]
Email: [EMAIL_1]


In [10]:
# rotina que faz OCR de PDF imagem
# por.traineddata https://github.com/tesseract-ocr/tessdata/tree/main
# salve este arquivo em D:/Users/abrantes/AppData/Local/Programs/Tesseract-OCR/tessdata
# consulte Udemy pdfimagem.ipynb
# pip install pdfplumber

import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import pdfplumber

def ocr_pdf_imagem(pdf_path):
    # Definir o caminho do executável do Tesseract (necessário apenas no Windows)
    pytesseract.pytesseract.tesseract_cmd = "C:/Program Files/Tesseract-OCR/tesseract.exe"
    pytesseract.pytesseract.tesseract_cmd = "D:/Users/abrantes/AppData/Local/Programs/Tesseract-OCR/tesseract.exe"
    
    # Caminho para o arquivo PDF
    # instalação do poppler https://github.com/oschwartz10612/poppler-windows/releases
    poppler_path = r'D:\Users\abrantes\poppler-24.08.0\Library\bin'  
    
    # Converter PDF em uma lista de imagens (cada página do PDF será uma imagem)
    pages = convert_from_path(pdf_path, 300, poppler_path=poppler_path)  # 300 DPI para qualidade de imagem
    
    # Percorrer cada página e extrair o texto
    text = ""
    for page in pages:
        # Converter a imagem para texto usando o pytesseract (OCR)
        text += pytesseract.image_to_string(page, lang='por')  # 'lang' define o idioma, 'por' para português
    
    # Exibir o texto extraído
    return text

def eh_pdf_imagem(caminho_pdf):
    with pdfplumber.open(caminho_pdf) as pdf:
        for pagina in pdf.pages:
            texto = pagina.extract_text()
            if texto and texto.strip():
                return False  # tem texto → não é só imagem
    return True  # nenhuma página tinha texto → é PDF imagem
    
pdf_path = 'pareceres/peticoes/PI0800882_29409161909986908_214.pdf'
teste = eh_pdf_imagem (pdf_path)
texto = ocr_pdf_imagem(pdf_path)
print(str(teste) + '#####' + texto)

False#####' INSTITUTO 8701 90092097
A DA PROPRIEDADE
INDUSTRIAL

Recurso de patente de invenção, modelo de utilidade ou certificado
de adição de invenção

Número do Processo: Pl 0800882-5

Dados do Depositante (71)
Depositante 1 de 1

Nome ou Razão Social: JOHN KURT JUNKERS
Tipo de Pessoa: Pessoa Fisica
CPF/CNPJ: US0001549173
Nacionalidade: Norte Americana
Qualificação Física: Engenheiro, arquiteto e afins
Endereço: 14 ALGONQUIN TRAIL, SADDLE RIVER, NEW JERSEY 07458 -
USA

Cidade:

Estado:

CEP:

País: Estados Unidos da América
Telefone:
Fax:

Email:

Referência Petição

Pedido : PIO800882-5

PETICIONAMENTO Eesta solicitação foi enviada pelo sistema Peticionamento Eletrônico em 16/09/2019 às
ELETRÔNICO 12:08, Petição 870190092097

Petição 870190092097, de 16/09/2019, pág. 1/11
Dados do Procurador

Procurador:

Nome ou Razão Social:
Numero OAB:

Numero API:
CPF/CNPUJ:

Endereço:

Cidade:

Estado:

CEP:

Telefone:

Fax:

Email:

Documentos anexados

Tipo Anexo

José Antonio de Souza Capp

In [14]:
import PyPDF2, os

arquivo = 'PI1004183_29409161953366073_214.pdf' # EXCELENTÍSSIMO SENHOR PRESIDENTE
arquivo = 'PI1004208_29409161920406602_214.pdf' # RECURSO CONTRA INDEFERIMENTO
arquivo = 'PI1004418_29409161918832101_214.pdf' # RECURSO AO INDEFERIMENTO
arquivo = 'PI1001299_29409161929090802_214.pdf' # RECURSO CONTRA INDEFERIMENTO
arquivo = 'PI1005286_29409161914398431_214.pdf' # RECURSO do despacho que indeferiu o Pedido
arquivo = 'PI1004831_29409161921403429_214.pdf' # apresentar recurso desta decisão PROBLEMA
arquivo = 'PI1003946_29409161937343447_214.pdf' # RECURSO CONTRA INDEFERIMENTO
arquivo = 'PI1001299_29409161929090802_214.pdf' # RECURSO CONTRA INDEFERIMENTO
arquivo = 'PI1005286_29409161914398431_214.pdf' # RECURSO do despacho que indeferiu o Pedido
arquivo = 'PI1003746_29409161938860178_214.pdf' # Em resposta ao Indeferimento
arquivo = 'PI1003760_29409161922238120_214.pdf' # para apresentar RECURSO CONTRA O INDEFERIMENTO
arquivo = 'PI1001007_29409161915941430_214.pdf' # RECURSO CONTRA O INDEFERIMENTO (9.2), DE ACORDO COM PUBLICAÇÃO
arquivo = 'PI1000672_29409161928888053_214.pdf' # ILUSTRÍSSIMOS EXAMINADORES DA DIRETORIA DE PATENTES
arquivo = 'PI0922619_29409161921988990_214.pdf' # RECURSO CONTRA INDEFERIMENTO
arquivo = 'PI0921237_29409161921173881_214.pdf' # RECURSO CONTRA INDEFERIMENTO
arquivo = 'PI0920800_29409161942731140_214.pdf' # RECURSO AO PRESIDENTE DO INPI
arquivo = 'PI0919841_29409161941803155_214.pdf' # RECURSO CONTRA INDEFERIMENTO
arquivo = 'PI0919693_29409161949692999_214.pdf' # RECURSO CONTRA O INDEFERIMENTO
arquivo = 'PI0919409_29409161924572914_214.pdf' # RECURSO CONTRA O INDEFERIMENTO
arquivo = 'PI0919236_29409161940400286_214.pdf' # RECURSO AO PRESIDENTE DO INPI
arquivo = 'PI0917535_29409161934542279_214.pdf' # RECURSO CONTRA O INDEFERIMENTO
arquivo = 'PI0914086_29409161941177793_214.pdf' # RECURSO CONTRA O INDEFERIMENTO exarado ao Pedido de Patente de Invenção
arquivo = 'PI0912882_29409161933242425_214.pdf' # RECURSO CONTRA O INDEFERIMENTO
arquivo = 'PI0911269_29409161919529933_214.pdf' # RECURSO CONTRA O INDEFERIMENTO
arquivo = 'PI0907077_29409161958795070_214.pdf' # RECURSO CONTRA O INDEFERIMENTO do PI
arquivo = 'PI0906612_29409161921906854_214.pdf' # RECURSO AO PRESIDENTE DO INPI
arquivo = 'PI0906181_29409161941424081_214.pdf' # Ilmo. Sr. Presidente do
arquivo = 'PI0817478_29409161938994123_214.pdf' # Recurso ao presidente do INPI
arquivo = 'PI0815327_29409161925862070_214.pdf' # RECURSO CONTRA O INDEFERIMENTO PI 0815327
arquivo = 'PI0811199_29409161915407060_214.pdf' # Recurso contra o indeferimento (9.2)
arquivo = 'PI0805704_29409161807942260_214.pdf' # ILMO SENHOR PRESIDENTE DO INSTITUTO NACIONAL DA
arquivo = 'PI0802657_29409161921852835_214.pdf' # Recurso que bastante faz 
arquivo = 'PI0800882_29409161909986908_214.pdf' # PDF imagem
arquivo = 'PI0716315_29409161921793251_214.pdf' # Recurso contra o indeferimento do pedido de
arquivo = 'PI0708406_29409161916850331_214.pdf' # Recurso contra a decisão de indeferimento
arquivo = 'PI0701396_29409161901216836_214.pdf' # PDF imagem, Interposição de recurso contra o indeferimento
arquivo = 'PI0608212_29409161803338023_214.pdf' # Recurso ao indeferimento
arquivo = 'MU9101438_29409161919978886_214.pdf' # Recurso contra o indeferimento
arquivo = 'MU9100724_29409162317696511_214.pdf' # Recurso ao indeferimento, gera varios espaços vazios porque tem imagens embutidas no texto
arquivo = 'MU9100581_29409162312309455_214.pdf' # Recurso contra o indeferimento

nome_sem_extensao = arquivo.replace('.pdf', '')
partes = nome_sem_extensao.split('_')
numero = partes[0]
numnossonumero = partes[1]

file_path = f"pareceres/peticoes/{numero}_{numnossonumero}_214.pdf"

all_text = ''
if os.path.exists(file_path):
    with open(file_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            all_text += page.extract_text() or ""
else:
    print(f"Arquivo não encontrado: {file_path}")
    
def anonimizacao(texto):
    texto_anon = ''
    anonymizer = DataAnonymizer()
    documento_normalizado = normalizar(texto) # elimina quebras de linha e caracteres especiais
    documento_inicial = anonymizer.iniciar_apos_recurso(documento_normalizado) # busca cabeçalho de início
    texto_anon = anonymizer.anonymize_texto(documento_inicial) # anonimiza referencias
    #print("=== DOCUMENTO ANONIMIZADO ===")
    if (texto_anon==''):
        documento_inicial = anonymizer.iniciar_apos_recurso_alternativo(documento_normalizado) # busca cabeçalho de início
        texto_anon = anonymizer.anonymize_texto(documento_inicial) # anonimiza referencias
    #print(texto_anon)
    
    ##if texto_anon == '':
        #print("Texto inicial não identificado")
        ##documento = ocr_pdf_imagem(file_path)
        ##documento_normalizado = normalizar(documento)
        ##documento_inicial = anonymizer.iniciar_apos_recurso(documento_normalizado)
        ##texto_anon = anonymizer.anonymize_texto(documento_inicial) # anonimiza referencias
        #print("=== DOCUMENTO ANONIMIZADO APOS OCR DA IMAGEM ===")
        ##if (texto_anon==''):
            ##documento_inicial = anonymizer.iniciar_apos_recurso_alternativo(documento_normalizado) # busca cabeçalho de início
            ##texto_anon = anonymizer.anonymize_texto(documento_inicial) # anonimiza referencias
        #print(texto_anon)
    return texto_anon

#print(all_text)
print(anonimizacao(all_text))


RECURSO CONTRA  O INDEFERIMENTO  
 
 
 
Ao pedido de Patente de Modelo de Utilidade  MU 9100581 -7, intitulado como 
DISTRIBUIDOR FLUTUANTE AUTOMATICO DE DISTRIBUICAO DE RACAO COM PRATO GIRATORIO 
APLICADO NA PISCICULTURA , pelos fatos e fundamentos adiante declinados:  
 
 
 
01  DA PROPOSICAO DO PRESENTE RECURSO:  
 
Tendo em vista o conteudo do indeferimento ora questionado, verifica -se a 
ocorrencia de uma serie de nao conformidades que levaram a proposicao das presentes Razoes de 
Recurso no sentido de salvaguardar os direitos d o RECORRENTE.  
 
Desta forma, nao ha como ser aceito tal entendimento, haja vista que existe uma 
serie de dissonancias entre os fatos narrados pelo Nobre Examinador. Em suma, o Parecer de 
Indeferimento  em questao esta dissociado da realidade fatica e juridica atinente a especie.  
  
Estas Razoes de Recurso possuem como objeto, o inteiro teor do indeferimento 
referente ao Processo MU 9100581 -7, publicado em 10/10/2023 , que foi dirigido em razao d o

In [16]:
import pandas as pd
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"select * from anterioridades_desc where resumo_recurso='' and numero in (select numero from carga) limit 2;"
comando = f"select * from anterioridades_desc where resumo_recurso='' and numero in (select numero from carga);"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
#print(resultado)
lista = df.values.tolist()
lista1 = lista
lista = df.iloc[:, 1].tolist()
lista.insert(0, 'numero')
print(lista)
#lista = [x for x in lista if x != "numero"]
#json_data = {"patents": [{"numero": item} for item in lista]}
#print(json_data)

['numero', '112012033666', '112014000466', '112015011774', '122019025423', '122020013634', '202012024436', '202013014629', 'PI1103692', '102013009882', '102014031235', '102016016661', '112014022817', '112015003727', '202012019591', 'MU9000951', 'PI0820657', '102012004873', '102014013268', '102015017470', '102019010055', '102022007897', '112012032204', '112013022907', '112014024641', '112015019446', '112016005389', '122020024337', '122020026570', '122020026576', '122021003228', '122021006465', '122021015578', '122022007908', '122022012256', '122022020636', '122023002765', '202012022499', '202013017528', '202014000433', '202014013394', '202014017096', '202014025663', '202020012873', 'PI0919841', 'PI1016173', 'PI1104976', '102012004079', '102012018962', '102012021084', '102012030377', '102012032591', '102012032927', '102013011905', '102013014262', '102013030975', '102014002008', '102014013800', '102014013993', '102014017842', '102014030105', '102015017609', '112012017243', '112012026999',

In [17]:
# ******************************************UPDATE ANTERIORIDADES_DESC CAMPO RESUMO_RECURSO 
# certifique-se de rodar a rotina acima conectar_siscap e de estar a VPN ligada

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts.prompt import PromptTemplate
import PyPDF2
import re

def limpar_caracteres_especiais(texto):
    # Remove caracteres não imprimíveis
    texto = re.sub(r'[^\x20-\x7EÀ-ÿ]', '', texto)
    return texto
    
def format_as_single_paragraph(text):
    # Remove quebras de linha e espaços extras
    formatted_text = ' '.join(line.strip() for line in text.splitlines() if line.strip())
    return formatted_text
    
load_dotenv(dotenv_path='.env', override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
url_openai = "https://api.openai.com/v1/chat/completions"

query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
json_data = conectar_siscap(url,return_json=True)
#json_data = {"patents": [{"numero": item} for item in lista]}
#json_data = json.dumps(json_data, indent=4, ensure_ascii=False)
data = json.loads(json_data)

data["patents"] = lista
#data["patents"] = ['numero','PI1005286','PI1001299'] # lista de numeros especificos

with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(data["patents"])):
        #if i==2: break
        #numero = data["patents"][i]["numero"]
        numero = data["patents"][i] # para ler a lista de numeros especificos
    
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where tipo_peticao='214' and numero='{numero}'" + '"'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        try:
            json_data = conectar_siscap(url,return_json=True)
            data1 = json.loads(json_data)
            numnossonumero = data1["patents"][0]["numnossonumero"]
            cd_imagem = data1["patents"][0]["cd_imagem"]
            
            file_path = f"pareceres/peticoes/{numero}_{numnossonumero}_214.pdf"
            print(file_path)
            
            all_text = ''
            with open(file_path, "rb") as file:
                reader = PyPDF2.PdfReader(file)
                num_pages = len(reader.pages)
                all_text = ""
                for page_num in range(num_pages):
                    page = reader.pages[page_num]
                    text = page.extract_text()
                    if text:
                        all_text += text
            #print(all_text)
            
            if all_text!='':
                all_text = limpar_caracteres_especiais(all_text)
                texto_anon = anonimizacao(all_text)
        
                url = "https://api.openai.com/v1/chat/completions"
                query = f"Resuma os principais argumentos apresentados pela requerente em sua petição recursal {texto_anon}. "
                data_json = {
                    "model": "gpt-5-mini",  # Use o modelo desejado, como 'gpt-4'
                    "messages": [
                        {"role": "user", "content": query}
                    ]
                }
                headers = {
                    "Authorization": f"Bearer {openai_api_key}",
                    "Content-Type": "application/json"
                }
                response = requests.post(url, headers=headers, json=data_json, verify=False)
                if response.status_code == 200:
                    resposta = response.json()
                    resumo = resposta['choices'][0]['message']['content']
                    resumo = resumo.replace("'","")
                    resumo = resumo.replace('"',"")
                    resumo = resumo.replace('“','')
                    resumo = resumo.replace('”','')
                    resumo = format_as_single_paragraph(resumo)
                    sql_resumo = f"UPDATE anterioridades_desc set resumo_recurso='{resumo}' WHERE numero='{numero}';"
                    print(sql_resumo)
                    f.write(sql_resumo + "\n")
                else:
                    print(f"Erro {response.status_code}: {response.text}")
                    
                #if i == 2:
                    #break
        except Exception as e:
            print(f"Não achei parecer de indeferimento {numero} {e}")

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM despachos_pag where tipo_peticao='214' and numero='112012033666'"


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


pareceres/peticoes/112012033666_29409161809558149_214.pdf


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.openai.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

UPDATE anterioridades_desc set resumo_recurso='Segue um resumo dos principais argumentos apresentados pela requerente (Sr. Robson Pessoa) no recurso contra o indeferimento: 1. Identificação e contexto processual - Requerente: Robson Pessoa, inventor e proprietário da oficina AUTO MECÂNICA ROBINHO E FILHO LTDA. - Alegação processual: tomou ciência do parecer técnico (despacho 7.1) mas não se manifestou no prazo porque, ao preparar o pedido, identificou erros formais e passou a estudar as Instruções Normativas (30/2013 e 31/2013). Esse estudo fez‑no perder o prazo de resposta, resultando no indeferimento, mas agora regularizou o pedido e preparou o recurso. 2. Regularizações e documentos apresentados - Afirma ter corrigido o relatório e as ilustrações (com numeração e identificação dos componentes) e adequou o pedido às instruções normativas vigentes. - Apresentou quadro reivindicatório reformulado e anexou relatório e desenhos atualizados para melhor compreensão da invenção. 3. Descriçã

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


pareceres/peticoes/102022016594_29409162341437750_214.pdf


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.openai.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


UPDATE anterioridades_desc set resumo_recurso='Posso resumir com precisão, mas preciso do texto da petição recursal (ou de trechos relevantes). Você pode colar aqui a petição ou indicar os pontos/trechos que quer que eu resuma. Enquanto isso, seguem duas coisas úteis: (1) modelo de resumo curto que uso normalmente; (2) lista das teses mais comuns em recursos, para você checar rapidamente quais aparecem na petição. 1) Modelo de resumo curto (use este formato se quiser um resumo pronto) - Partes e decisão atacada: quem recorre e qual decisão/trecho impugna. - Síntese dos fatos: 2–3 frases sobre o contexto fático relevante. - Fundamentos do recurso: 4–6 itens curtos com as teses jurídicas principais. - Pedido: o que a requerente pretende (anulação, reforma, novo julgamento, efeito suspensivo, etc.). Exemplo genérico de resumo (2–3 linhas por item) - Requerente: Fulana de Tal; decisão atacada: sentença que julgou improcedente o pedido de … . - Fatos: A requerente alega que … (resumo sucint